#  Data Collection

The question I wanted to answer here is whether consumer interest in fashion trends — the kind you can measure through Google searches — has any connection to how fashion stocks actually perform. When "quiet luxury" blows up on TikTok, does LVMH go up? When everyone's searching "zara haul", does Inditex move?

To check that, I need three things: stock prices, Google Trends data, and some macro context.

Stocks covered here:

| Brand | Ticker | Exchange |
|-------|--------|----------|
| LVMH | MC.PA | Euronext Paris |
| Inditex (Zara) | ITX.MC | Madrid |
| H&M | HM-B.ST | Stockholm |
| Coach / Tapestry | TPR | NYSE |

Sources:
1. yfinance - daily stock prices
2. pytrends - weekly Google search interest for "quiet luxury", "streetwear", "zara haul"
3. FRED - personal consumption expenditure and disposable income (macro context)



## 1. Setup

In [7]:
import os
import time
import warnings
import pandas as pd
import yfinance as yf
from pytrends.request import TrendReq

warnings.filterwarnings('ignore')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

TICKERS    = ['MC.PA', 'ITX.MC', 'HM-B.ST', 'TPR']
KEYWORDS   = ['quiet luxury', 'streetwear', 'zara haul']
START_DATE = '2020-01-01'
END_DATE   = '2025-12-31'

print(f'Tickers: {TICKERS}')
print(f'Keywords: {KEYWORDS}')
print(f'Period: {START_DATE} → {END_DATE}')

Tickers: ['MC.PA', 'ITX.MC', 'HM-B.ST', 'TPR']
Keywords: ['quiet luxury', 'streetwear', 'zara haul']
Period: 2020-01-01 → 2025-12-31


## 2. Stock Prices

Three of the four tickers are on European exchanges (Paris, Madrid, Stockholm), so the prices come back in EUR and SEK respectively. That's fine for now. The optimization runs on returns (percentage changes), which are currency-neutral, so we don't need to convert anything.

In [8]:
def download_prices(tickers, start, end):
    
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)

    # yfinance returns MultiIndex columns (Price, Ticker) for multiple tickers
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close'].copy()
    else:
        prices = raw[['Close']].copy()
        prices.columns = [tickers[0]]

    prices.index = pd.to_datetime(prices.index)

    missing = prices.columns[prices.isna().all()].tolist()
    if missing:
        print(f'[warn] No data for: {missing} — dropping.')
        prices.drop(columns=missing, inplace=True)

    return prices


prices = download_prices(TICKERS, START_DATE, END_DATE)
prices_clean = prices.ffill()

prices_clean.to_csv(os.path.join(DATA_DIR, 'prices.csv'))

print(f'Shape     : {prices_clean.shape}')
print(f'Date range: {prices_clean.index[0].date()} → {prices_clean.index[-1].date()}')
print(f'NaN cells : {prices_clean.isna().sum().sum()}')
print('Saved → data/prices.csv')
prices_clean.head()

Shape     : (1550, 4)
Date range: 2020-01-02 → 2025-12-30
NaN cells : 0
Saved → data/prices.csv


Ticker,HM-B.ST,ITX.MC,MC.PA,TPR
Date,,,,
2020-01-02,154.498077,25.925188,380.116302,23.393806
2020-01-03,152.398544,25.974180,380.071014,23.385117
2020-01-06,152.398544,25.745550,376.488800,23.680468
2020-01-07,154.578186,26.072166,377.259674,23.906330
2020-01-08,154.257660,26.129322,379.798950,24.062696


## 3. Google Trends via pytrends

Google Trends gives a weekly index (0 - 100) of search interest for any keyword. The number isn't absolute -  it's relative to the peak within the time range. So 100 means "highest search volume in this period", not "100 searches".

I'm tracking three keywords that map roughly to the stocks:
- "quiet luxury" → LVMH, Tapestry (aspirational, high-end)
- "streetwear" → H&M, broader market signal
- "zara haul" → Inditex directly

One thing to note: pytrends can get rate-limited by Google if you fire too many requests too fast. I'm pulling all three keywords in one batch to avoid that.

In [9]:
def fetch_trends(keywords, start, end, retries=3):

    timeframe = f'{start} {end}'
    pt = TrendReq(hl='en-US', tz=0, timeout=(10, 25))

    for attempt in range(1, retries + 1):
        try:
            pt.build_payload(keywords, timeframe=timeframe, geo='')
            raw = pt.interest_over_time()

            if raw.empty:
                raise ValueError('Google Trends returned empty response.')

            if 'isPartial' in raw.columns:
                raw = raw.drop(columns=['isPartial'])

            raw.index = pd.to_datetime(raw.index)
            return raw

        except Exception as e:
            print(f'[attempt {attempt}/{retries}] {e}')
            if attempt < retries:
                time.sleep(5 * attempt)

    raise RuntimeError('pytrends failed after all retries. Try again in a few minutes.')


trends = fetch_trends(KEYWORDS, START_DATE, END_DATE)

trends.to_csv(os.path.join(DATA_DIR, 'trends.csv'))

print(f'Shape     : {trends.shape}  (weekly observations × keywords)')
print(f'Date range: {trends.index[0].date()} → {trends.index[-1].date()}')
print('Saved → data/trends.csv')
trends.head(10)

Shape     : (72, 3)  (weekly observations × keywords)
Date range: 2020-01-01 → 2025-12-01
Saved → data/trends.csv


,quiet luxury,streetwear,zara haul
date,,,
2020-01-01,0,32,0
2020-02-01,0,33,0
2020-03-01,0,30,0
2020-04-01,0,35,0
2020-05-01,0,35,1
2020-06-01,0,38,1
2020-07-01,0,43,1
2020-08-01,0,43,1
2020-09-01,0,40,1


## 4. Macro Data from FRED

Fashion spending is pretty sensitive to how much money people actually have. When disposable income drops, discretionary spending (especially luxury) tends to follow. I'm pulling two FRED series as macro context:

### Series &  What is it? 

- PCE -  Personal Consumption Expenditures — total consumer spending (monthly)
- DSPIC96 - Real Disposable Personal Income — income after taxes, inflation-adjusted (monthly) 

No API key needed - FRED publishes everything as a plain public CSV.

In [ ]:
FRED_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'

def fetch_fred(series_id, start, end):
    """Read a FRED series from the public CSV endpoint — no API key required."""
    url = FRED_URL.format(series_id=series_id)
    interest  = pd.read_csv(url, index_col=0, na_values='.')
    interest.index = pd.to_datetime(interest.index)
    s = interest.iloc[:, 0].loc[start:end].dropna()
    s.name = series_id
    return s


pce    = fetch_fred('PCE',     START_DATE, END_DATE)
income = fetch_fred('DSPIC96', START_DATE, END_DATE)

macro = pd.DataFrame({'pce': pce, 'disposable_income': income})
macro.to_csv(os.path.join(DATA_DIR, 'macro.csv'))

print(f'PCE: {len(pce)} obs  ({pce.index[0].date()} → {pce.index[-1].date()})')
print(f'DSPIC96: {len(income)} obs  ({income.index[0].date()} → {income.index[-1].date()})')
macro.tail()

PCE      : 72 obs  (2020-01-01 → 2025-12-01)
DSPIC96  : 72 obs  (2020-01-01 → 2025-12-01)
Saved → data/macro.csv


,pce,disposable_income
observation_date,,
2025-08-01,21123.8,18076.9
2025-09-01,21202.4,18094.6
2025-10-01,21288.1,18063.1
2025-11-01,21356.0,18078.6
2025-12-01,21445.9,18070.6
